# Deep Learning Extractive Summarizer

This notebook contains both workflows:

- Train the TensorFlow extractive summarizer.
- Evaluate an already trained `.keras` model without retraining.

The model is extractive: it scores article sentences and selects the highest scoring sentences as the summary.


## 1. Setup


In [ ]:
import math
import random
import re
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

SEED = 42
CSV_PATH = 'cleaned_data.csv'
ARTICLE_COL = 'article'
SUMMARY_COL = 'highlights'

MODEL_DIR = Path('runs/tf_extractive_v2')
BEST_MODEL_PATH = MODEL_DIR / 'best_sentence_ranker.keras'
FINAL_MODEL_PATH = MODEL_DIR / 'final_sentence_ranker.keras'

# If you are on Linux/WSL, the original model files also work:
# BEST_MODEL_PATH = MODEL_DIR / 'best_sentence_ranker.keras'
# FINAL_MODEL_PATH = MODEL_DIR / 'final_sentence_ranker.keras'

MAX_VOCAB = 50000
MAX_SENTENCE_TOKENS = 80
MAX_SENTENCES_PER_ARTICLE = 80
SUMMARY_SENTENCES = 3
BATCH_SIZE = 256
EPOCHS = 20

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2. Load Data And Create The Same Split

The split is deterministic because it uses the same seed every time. This keeps evaluation comparable across runs.


In [ ]:
try:
    df = pd.read_csv(CSV_PATH, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding='latin1')

df = df[[ARTICLE_COL, SUMMARY_COL]].dropna().copy()
df[ARTICLE_COL] = df[ARTICLE_COL].astype(str).str.strip()
df[SUMMARY_COL] = df[SUMMARY_COL].astype(str).str.strip()
df = df[(df[ARTICLE_COL] != '') & (df[SUMMARY_COL] != '')]
df = df.drop_duplicates(subset=[ARTICLE_COL]).reset_index(drop=True)

indices = np.arange(len(df))
np.random.default_rng(SEED).shuffle(indices)

test_n = int(len(df) * 0.10)
val_n = int(len(df) * 0.10)

test_df = df.iloc[indices[:test_n]].reset_index(drop=True)
val_df = df.iloc[indices[test_n:test_n + val_n]].reset_index(drop=True)
train_df = df.iloc[indices[test_n + val_n:]].reset_index(drop=True)

print('train:', len(train_df), 'validation:', len(val_df), 'test:', len(test_df))
df.head()


## 3. Helper Functions

These functions are needed for both training and evaluation.


In [ ]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])")


def words(text):
    return [m.group(0).lower() for m in WORD_RE.finditer(str(text))]


def split_sentences(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    sentences = [s.strip() for s in SENTENCE_RE.split(text) if len(words(s)) >= 5]
    return (sentences or [text])[:MAX_SENTENCES_PER_ARTICLE]


def rouge_n_f1(prediction, reference, n):
    pred_tokens = words(prediction)
    ref_tokens = words(reference)
    if len(pred_tokens) < n or len(ref_tokens) < n:
        return 0.0

    pred = Counter(tuple(pred_tokens[i:i + n]) for i in range(len(pred_tokens) - n + 1))
    ref = Counter(tuple(ref_tokens[i:i + n]) for i in range(len(ref_tokens) - n + 1))
    overlap = sum((pred & ref).values())
    if overlap == 0:
        return 0.0

    precision = overlap / max(sum(pred.values()), 1)
    recall = overlap / max(sum(ref.values()), 1)
    return 2 * precision * recall / (precision + recall)


def lcs_length(left, right):
    previous = [0] * (len(right) + 1)
    for token_left in left:
        current = [0]
        for j, token_right in enumerate(right, start=1):
            if token_left == token_right:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current
    return previous[-1]


def rouge_l_f1(prediction, reference):
    pred_tokens = words(prediction)
    ref_tokens = words(reference)
    if not pred_tokens or not ref_tokens:
        return 0.0

    overlap = lcs_length(pred_tokens, ref_tokens)
    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def rouge_scores(prediction, reference):
    return {
        'rouge1': rouge_n_f1(prediction, reference, 1),
        'rouge2': rouge_n_f1(prediction, reference, 2),
        'rougeL': rouge_l_f1(prediction, reference),
    }


def sentence_features(sentences):
    total = len(sentences)
    denom = max(total - 1, 1)
    rows = []

    for i, sentence in enumerate(sentences):
        position = i / denom
        rows.append([
            position,
            1.0 - position,
            min(len(words(sentence)) / MAX_SENTENCE_TOKENS, 2.0),
            min(total / 80.0, 2.0),
            1.0 if i < 3 else 0.0,
            math.exp(-i / 4.0),
        ])

    return np.asarray(rows, dtype='float32')


## 4. Training Data Creation

Run this section only when training.

Each sentence gets a binary label:

- `1`: useful sentence for the summary.
- `0`: not useful.

The label is created from ROUGE-1 overlap between the article sentence and the reference highlight.


In [ ]:
def build_sentence_examples(dataframe):
    sentence_texts = []
    feature_rows = []
    labels = []

    for row in dataframe.to_dict('records'):
        sentences = split_sentences(row[ARTICLE_COL])
        reference = row[SUMMARY_COL]
        scores = np.asarray([rouge_n_f1(sentence, reference, 1) for sentence in sentences])

        positive_idx = set(np.argsort(scores)[::-1][:3])
        positive_idx.update(np.where(scores >= 0.15)[0])

        features = sentence_features(sentences)
        for i, sentence in enumerate(sentences):
            sentence_texts.append(sentence)
            feature_rows.append(features[i])
            labels.append(1.0 if i in positive_idx and scores[i] > 0 else 0.0)

    return (
        np.asarray(sentence_texts, dtype=str),
        np.asarray(feature_rows, dtype='float32'),
        np.asarray(labels, dtype='float32').reshape(-1, 1),
    )


X_text_train, X_feat_train, y_train = build_sentence_examples(train_df)
X_text_val, X_feat_val, y_val = build_sentence_examples(val_df)

print('train sentence examples:', len(X_text_train))
print('validation sentence examples:', len(X_text_val))
print('positive label rate:', float(y_train.mean()))


## 5. Build The Deep Learning Model

Run this section only when training.

The model is trained from scratch:

- `TextVectorization` learns vocabulary from the training data.
- `Embedding` starts randomly and learns during training.
- `BiLSTM` encodes each sentence.
- Dense layers output one probability: should this sentence be selected?

`binary_crossentropy` is used because each sentence label is binary: selected or not selected.


In [ ]:
vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    standardize='lower_and_strip_punctuation',
    output_mode='int',
    output_sequence_length=MAX_SENTENCE_TOKENS,
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_text_train).batch(BATCH_SIZE))

sentence_input = keras.Input(shape=(), dtype=tf.string, name='sentence')
feature_input = keras.Input(shape=(6,), dtype=tf.float32, name='features')

x = vectorizer(sentence_input)
x = keras.layers.Embedding(MAX_VOCAB, 96, mask_zero=True)(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(96))(x)
x = keras.layers.Concatenate()([x, feature_input])
x = keras.layers.Dense(96, activation='relu')(x)
x = keras.layers.Dropout(0.45)(x)
output = keras.layers.Dense(1, activation='sigmoid')(x)

model = keras.Model([sentence_input, feature_input], output)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='pr_auc'), 'accuracy'],
)

model.summary()


## 6. Train And Save

Run this section only when training.

The notebook saves:

- `best_sentence_ranker.keras`: best validation PR-AUC checkpoint.
- `final_sentence_ranker.keras`: final model after training.


In [ ]:
def make_dataset(texts, features, labels=None, weights=None, shuffle=False):
    inputs = {'sentence': texts, 'features': features}
    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices(inputs)
    elif weights is None:
        ds = tf.data.Dataset.from_tensor_slices((inputs, labels))
    else:
        ds = tf.data.Dataset.from_tensor_slices((inputs, labels, weights))

    if shuffle:
        ds = ds.shuffle(min(len(texts), 20000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


positive = max(float((y_train == 1).sum()), 1.0)
negative = max(float((y_train == 0).sum()), 1.0)
positive_weight = min(negative / positive, 12.0)
sample_weights = np.where(y_train == 1, positive_weight, 1.0).astype('float32')

train_ds = make_dataset(X_text_train, X_feat_train, y_train, sample_weights, shuffle=True)
val_ds = make_dataset(X_text_val, X_feat_val, y_val)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_pr_auc', mode='max', save_best_only=True),
    keras.callbacks.CSVLogger(str(MODEL_DIR / 'training_log.csv')),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
model.save(FINAL_MODEL_PATH)


## 7. Evaluation Only: Trained TensorFlow Model

Run this section for evaluation. It loads the saved `.keras` model and reports only the trained TensorFlow model metrics.


In [ ]:
def make_encoding_compatible_keras_copy(model_path):
    model_path = Path(model_path)
    safe_path = model_path.with_name(model_path.stem + '_compatible.keras')

    if safe_path.exists() and safe_path.stat().st_mtime >= model_path.stat().st_mtime:
        return safe_path

    with zipfile.ZipFile(model_path, 'r') as src, zipfile.ZipFile(safe_path, 'w', compression=zipfile.ZIP_DEFLATED) as dst:
        for info in src.infolist():
            data = src.read(info.filename)
            if info.filename.endswith('vocabulary.txt'):
                vocab = data.decode('utf-8', errors='replace').splitlines()
                safe_vocab = []
                for i, token in enumerate(vocab):
                    safe_vocab.append(f'nonascii_token_{i}' if token and not token.isascii() else token)
                data = '\n'.join(safe_vocab).encode('ascii')
            dst.writestr(info, data)

    return safe_path


def load_model_safely(model_path):
    try:
        return keras.models.load_model(model_path, compile=False)
    except Exception as error:
        message = str(error)
        if 'charmap' not in message and 'codec' not in message and 'TextVectorization' not in message:
            raise
        safe_path = make_encoding_compatible_keras_copy(model_path)
        print('Loading encoding-compatible copy:', safe_path)
        return keras.models.load_model(safe_path, compile=False)


if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {BEST_MODEL_PATH}')

model = load_model_safely(BEST_MODEL_PATH)
print('Loaded model:', BEST_MODEL_PATH)


def make_inference_dataset(texts, features):
    inputs = {'sentence': texts, 'features': features}
    return tf.data.Dataset.from_tensor_slices(inputs).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def summarize_article(article, model, k=3, lead_bias=0.20):
    sentences = split_sentences(article)
    features = sentence_features(sentences)
    ds = make_inference_dataset(np.asarray(sentences, dtype=str), features)

    learned_scores = model.predict(ds, verbose=0).reshape(-1)
    lead_scores = np.exp(-np.arange(len(sentences)) / 4.0)
    scores = (1.0 - lead_bias) * learned_scores + lead_bias * lead_scores

    top_idx = sorted(np.argsort(scores)[::-1][:k])
    return ' '.join(sentences[i] for i in top_idx)


def evaluate_model(dataframe):
    rows = []
    for row in dataframe.to_dict('records'):
        generated = summarize_article(row[ARTICLE_COL], model, k=SUMMARY_SENTENCES)
        rows.append({
            'reference': row[SUMMARY_COL],
            'generated': generated,
            **rouge_scores(generated, row[SUMMARY_COL]),
        })

    results = pd.DataFrame(rows)
    metrics = results[['rouge1', 'rouge2', 'rougeL']].agg(['mean', 'median'])
    return metrics, results


model_metrics, model_results = evaluate_model(test_df)
display(model_metrics)

model_results.to_csv(MODEL_DIR / 'notebook_test_predictions.csv', index=False)
model_results.head()


## 8. Inspect One Prediction

Run this after evaluation if you want to see one generated summary.


In [ ]:
article_index = 0
article = test_df.loc[article_index, ARTICLE_COL]
reference = test_df.loc[article_index, SUMMARY_COL]
generated = summarize_article(article, model)

print('Generated summary:\n')
print(generated)
print('\nReference summary:\n')
print(reference)
print('\nROUGE:')
print(rouge_scores(generated, reference))
